# WMIS Surface Water Quality Scraper — v3

Major change from earlier versions: instead of discovering stations through the
slow/timeout-prone `webservice.exe` functions, we now use
`https://data.water.vic.gov.au/WMIS/data/anon/internet/stations/stations.json` —
a single static JSON file with full metadata for every station on the network,
including:

- `sitetype` / `sitetype_decode` — lets us filter to **surface water only**,
  excluding groundwater bores (which is what a blind numeric scan kept hitting
  first).
- `parametertype` — a list of every parameter code tracked at that station.
  Codes ending in `wq` appear to be the quality-checked/lab variant (e.g.
  `totalalkalinityascaco3wq`) versus a raw telemetry variant without the
  suffix (e.g. `borewaterlevelahd`) — this is the "quality filter" distinction.

We still use the proven `webservice.exe` `get_ts_traces` / `get_variable_list`
calls for actually fetching time series data and the authoritative
`"Available for release"` flag — `stations.json` is used to pick good
candidate stations fast, not to fetch data itself.

Run top to bottom. Step 2 is important — it prints the real `sitetype` values
and the real surface-water `parametertype` slugs, since we're confirming
these empirically rather than guessing.

## Step 0 — Imports & config

In [24]:
import json
import os
import time
from datetime import datetime

import pandas as pd
import requests

BASE_URL = "https://data.water.vic.gov.au/WMIS/cgi/webservice.exe"
STATIONS_JSON_URL = "https://data.water.vic.gov.au/WMIS/data/anon/internet/stations/stations.json"

START_TIME = "20230101000000"
END_TIME = "20241231235959"

OUTPUT_DIR = os.path.expanduser("~/Desktop/wmis_export")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR, "| exists:", os.path.exists(OUTPUT_DIR))

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
LONG_CSV = os.path.join(OUTPUT_DIR, "wmis_all_stations_long.csv")
SKIPPED_LOG = os.path.join(OUTPUT_DIR, "skipped_stations.log")
FAILURES_LOG = os.path.join(OUTPUT_DIR, "failures.log")
STATIONS_CACHE = os.path.join(OUTPUT_DIR, "stations_raw_cache.json")

REQUEST_DELAY = 0.4
DEFAULT_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF = 2.0

DATASOURCE = "WQ"  # confirmed - all 7 parameters live here

# Confirmed against station 225210's real variable list (a known surface water
# gauge). Do not switch to keyword matching for these 7 - it previously
# produced false matches (Alkalinity onto pH, Salinity onto TDS).
CONFIRMED_PARAM_CODES = {
    "pH": "210.00",
    "Streamflow": "141.00",
    "Stream Water Level": "100.00",
    "Salinity (EC)": "820.00",
    "Water Temperature": "450.00",
    "Turbidity": "810.00",
    "Alkalinity": "2113.00",   # Total Alkalinity as CaCO3 (mg/L) - not all stations have this
}

print("Config loaded.")

Output dir: /Users/harsh/Desktop/wmis_export | exists: True
Config loaded.


## Step 1 — Core request function (with retry/backoff)

In [25]:
def hydstra_request(payload: dict, timeout: int = DEFAULT_TIMEOUT) -> dict:
    """
    Calls webservice.exe. The JSON payload must be appended to the URL RAW
    (unencoded braces/quotes/commas) - confirmed working, do not url-encode
    or pass via requests' params= kwarg.
    """
    query = json.dumps(payload, separators=(",", ":"))
    url = f"{BASE_URL}?{query}"

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            data = resp.json()
            if data.get("error_num", 0) != 0:
                raise RuntimeError(f"API error {data['error_num']}: {data.get('error_msg')}")
            return data
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF * attempt
                print(f"  retrying ({attempt}/{MAX_RETRIES}) after error: {e}")
                time.sleep(wait)
            else:
                raise last_err
    raise last_err

print("hydstra_request ready")

hydstra_request ready


## Step 2 — Download the full station metadata file

This is a plain static JSON file (not the flaky `webservice.exe` API), so it
shouldn't hit the 504 timeouts we saw before. It's large, so this may take
a little while. Cached to disk after the first successful download so re-runs
are instant.

In [26]:
if os.path.exists(STATIONS_CACHE):
    print("Loading cached copy...")
    with open(STATIONS_CACHE) as f:
        raw_stations = json.load(f)
else:
    print("Downloading stations.json (this may take a minute)...")
    resp = requests.get(STATIONS_JSON_URL, timeout=180)
    resp.raise_for_status()
    raw_stations = resp.json()
    with open(STATIONS_CACHE, "w") as f:
        json.dump(raw_stations, f)
    print("Downloaded and cached.")

print(f"{len(raw_stations)} total station records")

Loading cached copy...
7910 total station records


## Step 3 — Inspect site types and confirm the surface-water filter

We know `"GW"` = groundwater from what we've already seen. Let's confirm the
actual surface-water code(s) rather than guessing.

In [27]:
sitetype_counts = {}
for rec in raw_stations:
    key = (rec.get("sitetype"), rec.get("sitetype_decode"))
    sitetype_counts[key] = sitetype_counts.get(key, 0) + 1

for key, count in sorted(sitetype_counts.items(), key=lambda x: -x[1]):
    print(f"{count:6d}  {key}")

  6138  ('GW', 'Groundwater site')
  1772  ('SW', 'Surface water site')


## Step 4 — Build the filtered, cleaned surface-water station list

Edit `SURFACE_WATER_SITETYPES` below based on what Step 3 printed (defaults to
excluding `"GW"` and keeping everything else, which is a safe starting point
even if the exact surface-water code differs from what we expect).

In [28]:
SURFACE_WATER_SITETYPES = None  # None = "anything that is not GW"; or set explicitly e.g. {"SW"}

def is_surface_water(rec):
    st = rec.get("sitetype")
    if SURFACE_WATER_SITETYPES is not None:
        return st in SURFACE_WATER_SITETYPES
    return st != "GW"

surface_records = [r for r in raw_stations if is_surface_water(r)]
print(f"{len(surface_records)} surface-water station records "
      f"(out of {len(raw_stations)} total)")

# --- Auto-detect the real lat/lon field names in stations.json ---
# WMIS exports aren't consistent about this, so we probe real records
# instead of hardcoding a guess that might be wrong.
LAT_KEY_CANDIDATES = ["latitude", "lat", "Latitude", "decimal_latitude", "y", "site_latitude"]
LON_KEY_CANDIDATES = ["longitude", "long", "lon", "Longitude", "decimal_longitude", "x", "site_longitude"]

def _find_key(records, candidates):
    for key in candidates:
        for r in records:
            if r.get(key) not in (None, ""):
                return key
    return None

LAT_KEY = _find_key(surface_records, LAT_KEY_CANDIDATES)
LON_KEY = _find_key(surface_records, LON_KEY_CANDIDATES)
print(f"Detected latitude field: {LAT_KEY!r}, longitude field: {LON_KEY!r}")
if LAT_KEY is None or LON_KEY is None:
    print("WARNING: couldn't auto-detect lat/lon fields. Inspect the keys below "
          "and set LAT_KEY / LON_KEY manually:")
    print(sorted(surface_records[0].keys()) if surface_records else "no records")

stations_df = pd.DataFrame([{
    "station": r.get("station"),
    "station_name": r.get("station_name"),
    "sitetype": r.get("sitetype"),
    "sitetype_decode": r.get("sitetype_decode"),
    "basin_decode": r.get("basin_decode"),
    "active": r.get("active"),
    "latitude": r.get(LAT_KEY) if LAT_KEY else None,
    "longitude": r.get(LON_KEY) if LON_KEY else None,
    "parametertype": r.get("parametertype", []),
} for r in surface_records])

stations_df["station"] = stations_df["station"].astype(str).str.strip()
stations_df = stations_df[stations_df["station"].notna()]
stations_df = stations_df[stations_df["station"] != ""]
stations_df = stations_df[stations_df["station"] != "0"]
stations_df = stations_df.drop_duplicates(subset="station")

stations_df["latitude"] = pd.to_numeric(stations_df["latitude"], errors="coerce")
stations_df["longitude"] = pd.to_numeric(stations_df["longitude"], errors="coerce")

# Lookup used in Step 12 to attach name/lat/lon to every output row
STATION_META = stations_df.set_index("station")[["station_name", "latitude", "longitude"]].to_dict("index")

print(f"{len(stations_df)} stations after cleanup")
stations_df.head(10)

1772 surface-water station records (out of 7910 total)
Detected latitude field: None, longitude field: None
['active', 'aliasid', 'artesian', 'basin', 'basin_decode', 'catcharea', 'catcharea_decode', 'cease', 'cease_decode', 'commence', 'commence_decode', 'condition', 'conditiondate', 'contcode', 'contcode_decode', 'control', 'ctf', 'datum', 'datum_decode', 'easting', 'elev', 'elevacc', 'elevacc_decode', 'gaugcount', 'gauge', 'gaugmaxdate', 'gaugmindate', 'grdatum', 'grdatum_decode', 'headworksheight', 'latitude_decode', 'lldatum', 'lldatum_decode', 'longitude_decode', 'maxdailyrainfall', 'maxdailyrainfalldate', 'maxgaug', 'maxgdate', 'maxgdate_decode', 'maxsdate', 'maxstage', 'monitortype', 'mwadb', 'northing', 'numpoints', 'object_type', 'owner', 'parametertype', 'periodend', 'periodstart', 'posacc', 'posacc_decode', 'region_decode', 'rlmp', 'rlmpcomments', 'rlmpdate', 'screenbot', 'screenedlithology', 'screentop', 'site_groups', 'site_no', 'sitetype', 'sitetype_decode', 'spillway', 

,station,station_name,sitetype,sitetype_decode,basin_decode,active,latitude,longitude,parametertype
0,221001,GENOA RIVER @ ROCKTON,SW,Surface water site,East Gippsland,false,NaN,NaN,"[airtemperaturewq, colourtruefilteredpcuwq, di..."
1,221200,BEMM RIVER @ CLUB TERRACE,SW,Surface water site,East Gippsland,false,NaN,NaN,[streamflowmeandaily]
2,221201,CANN RIVER (WEST BRANCH) @ WEERAGUA,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, alkalinityashardnessmeasure..."
3,221202,GENOA RIVER @ WANGARABELL,SW,Surface water site,East Gippsland,true,NaN,NaN,"[streamflow, streamflowmeandaily, streamwaterl..."
4,221203,BETKA RIVER @ MALLACOOTA,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]"
5,221204,THURRA RIVER @ POINT HICKS,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamflowmeandaily, streamwaterl..."
6,221205,BEMM RIVER @ BEMM RIVER,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]"
7,221206,CANN RIVER @ NOORINBEE,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]"
8,221207,ERRINUNDRA RIVER @ ERRINUNDRA,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, colourtruefilteredpcuwq, di..."
9,221208,WINGAN RIVER @ WINGAN INLET NATIONAL PARK,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, chlorophyllawq, colourtruef..."


## Step 5 — See what parameter slugs actually exist at surface-water stations

`parametertype` uses different naming than the Hydstra numeric codes we
already confirmed (e.g. `phwq`, `totalalkalinityascaco3wq`). This is just for
our own reference/pre-filtering — printed here so we can see real values
instead of guessing.

In [29]:
all_slugs = set()
for plist in stations_df["parametertype"]:
    all_slugs.update(plist)

print(f"{len(all_slugs)} distinct parameter slugs across surface-water stations\n")
for slug in sorted(all_slugs):
    print(slug)

84 distinct parameter slugs across surface-water stations

airtemperature
airtemperaturewq
alkalinityasbicarbonatecaco3wq
alkalinityascarbonatecaco3wq
alkalinityashardnesscalculatedcaco3wq
alkalinityashardnessmeasuredcaco3wq
alkalinityashydroxidecaco3wq
alkalinitytotalcaco3wq
barometricpressure
berylliumwq
calciumtotalwq
chlorideasclwq
chlorophylla
chlorophyllawq
colourtruefilteredpcuwq
dissolvedorganiccarbondocwq
dissolvedoxygendo
dissolvedoxygendoadditionalsensor
dissolvedoxygendofieldwq
dissolvedoxygendolabwq
fluorideasfwq
ionicbalancewq
measuredflow
nitrogenasammoniatotalwq
nitrogenasnitrateno2wq
nitrogenasnitriteno3wq
nitrogenasnoxwq
nitrogenastotalkjeldahltknwq
nitrogenastotalwq
ph
phadditionalsensor
phaeophytinwq
phlabwq
phosphorusasfrpwq
phosphorusastrpwq
phosphorustotalwq
photosynradiation
phwq
potassiumaskwq
rainfall
relativehumidity
salinityasec25labwq
salinityasec25wq
salinityaselectricalconductivity25additionalsensor
salinityaselectricalconductivityec
salinityaselectricalc

## Step 6 — Fast pre-filter using those slugs

Skips calling the (slower) `get_variable_list` API entirely for stations whose
`parametertype` list obviously doesn't contain anything relevant to our 7
parameters — a big speedup across thousands of stations. This is a coarse
pre-filter only; the actual authoritative check still happens in Step 8 via
the live API and the `"Available for release"` flag.

**Check the printed slugs from Step 5 and adjust the keyword lists below if
anything looks off** — e.g. if streamflow/level use different wording than
guessed here.

In [30]:
PARAM_SLUG_HINTS = {
    "pH": ["phwq", "phlabwq"],
    "Streamflow": ["streamdischarge", "discharge"],
    "Stream Water Level": ["streamwaterlevel", "waterlevel"],
    "Salinity (EC)": ["salinityasec25"],
    "Water Temperature": ["watertemperature"],
    "Turbidity": ["turbidityntu"],
    "Alkalinity": ["totalalkalinity"],
}

def likely_has_any_target_param(parametertype_list):
    slugs_lower = [s.lower() for s in parametertype_list]
    for param, hints in PARAM_SLUG_HINTS.items():
        for hint in hints:
            if any(hint in s for s in slugs_lower):
                return True
    return False

stations_df["likely_relevant"] = stations_df["parametertype"].apply(likely_has_any_target_param)
n_relevant = stations_df["likely_relevant"].sum()
print(f"{n_relevant} / {len(stations_df)} surface-water stations "
      f"look like they have at least one of our 7 parameters")

candidate_stations_df = stations_df[stations_df["likely_relevant"]].copy()
candidate_stations_df.head(10)

1460 / 1712 surface-water stations look like they have at least one of our 7 parameters


,station,station_name,sitetype,sitetype_decode,basin_decode,active,latitude,longitude,parametertype,likely_relevant
0,221001,GENOA RIVER @ ROCKTON,SW,Surface water site,East Gippsland,false,NaN,NaN,"[airtemperaturewq, colourtruefilteredpcuwq, di...",True
2,221201,CANN RIVER (WEST BRANCH) @ WEERAGUA,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, alkalinityashardnessmeasure...",True
3,221202,GENOA RIVER @ WANGARABELL,SW,Surface water site,East Gippsland,true,NaN,NaN,"[streamflow, streamflowmeandaily, streamwaterl...",True
4,221203,BETKA RIVER @ MALLACOOTA,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]",True
5,221204,THURRA RIVER @ POINT HICKS,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamflowmeandaily, streamwaterl...",True
6,221205,BEMM RIVER @ BEMM RIVER,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]",True
7,221206,CANN RIVER @ NOORINBEE,SW,Surface water site,East Gippsland,false,NaN,NaN,"[streamflow, streamwaterlevel]",True
8,221207,ERRINUNDRA RIVER @ ERRINUNDRA,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, colourtruefilteredpcuwq, di...",True
9,221208,WINGAN RIVER @ WINGAN INLET NATIONAL PARK,SW,Surface water site,East Gippsland,true,NaN,NaN,"[airtemperaturewq, chlorophyllawq, colourtruef...",True
10,221209,CANN RIVER (EAST BRANCH) @ WEERAGUA,SW,Surface water site,East Gippsland,true,NaN,NaN,"[dissolvedoxygendofieldwq, phwq, rainfall, sal...",True


## Step 7 — Variable matching (confirmed codes + 'Available for release' filter)

In [31]:
def match_parameters_for_station(station: str):
    """
    Returns (matches, skip_reason).
    matches: list of dicts {parameter, variable_code, datasource, matched_name}
    skip_reason: None if matches found, otherwise a short string explaining why
                 nothing was found (for logging).
    """
    try:
        data = hydstra_request({
            "function": "get_variable_list",
            "version": 1,
            "params": {"site_list": station, "datasource": DATASOURCE},
        })
    except Exception as e:
        return [], f"get_variable_list error: {e}"

    sites = data.get("return", {}).get("sites", [])
    if not sites:
        return [], "no site entry returned"

    variables = sites[0].get("variables", [])
    if not variables:
        return [], "no WQ variables available at this station"

    # Only consider variables actually available for release
    available_codes = {
        v["variable"]: v["name"]
        for v in variables
        if "variable" in v and v.get("subdesc", "").strip().lower() == "available for release"
    }

    if not available_codes:
        return [], "WQ variables present but none marked 'Available for release'"

    matches = []
    for param, code_ in CONFIRMED_PARAM_CODES.items():
        if code_ in available_codes:
            matches.append({
                "parameter": param,
                "variable_code": code_,
                "datasource": DATASOURCE,
                "matched_name": available_codes[code_],
            })

    if not matches:
        return [], "WQ data present but none of the 7 target parameters found"

    return matches, None

print("match_parameters_for_station ready")

match_parameters_for_station ready


## Step 8 — Test the matcher on the known-good station

In [32]:
test_matches, skip_reason = match_parameters_for_station("225210")
if skip_reason:
    print("Skip reason:", skip_reason)
else:
    for m in test_matches:
        print(m)

{'parameter': 'pH', 'variable_code': '210.00', 'datasource': 'WQ', 'matched_name': 'Acidity/Alkalinity (pH)'}
{'parameter': 'Streamflow', 'variable_code': '141.00', 'datasource': 'WQ', 'matched_name': 'Stream Discharge (ML/d)'}
{'parameter': 'Stream Water Level', 'variable_code': '100.00', 'datasource': 'WQ', 'matched_name': 'Stream Water Level (m)'}
{'parameter': 'Salinity (EC)', 'variable_code': '820.00', 'datasource': 'WQ', 'matched_name': 'Conductivity (µS/cm)'}
{'parameter': 'Water Temperature', 'variable_code': '450.00', 'datasource': 'WQ', 'matched_name': 'Water Temperature (°C)'}
{'parameter': 'Turbidity', 'variable_code': '810.00', 'datasource': 'WQ', 'matched_name': 'Turbidity (NTU)'}


## Step 9 — Time series fetch function

In [33]:
def get_timeseries(station: str, variable_code: str, datasource: str = DATASOURCE) -> pd.DataFrame:
    payload = {
        "function": "get_ts_traces",
        "version": 2,
        "params": {
            "site_list": station,
            "datasource": datasource,
            "varfrom": variable_code,
            "varto": variable_code,
            "start_time": START_TIME,
            "end_time": END_TIME,
            "interval": "day",
            "data_type": "mean",
            "multiplier": "1",
        },
    }
    data = hydstra_request(payload)
    traces = data.get("return", {}).get("traces", [])
    if not traces:
        return pd.DataFrame()
    points = traces[0].get("trace", [])
    df = pd.DataFrame(points)
    if df.empty:
        return df
    df = df.rename(columns={"v": "value", "t": "datetime", "q": "quality_code"})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["datetime"] = pd.to_datetime(df["datetime"].astype(str), format="%Y%m%d%H%M%S")
    return df[["datetime", "value", "quality_code"]]

print("get_timeseries ready")

get_timeseries ready


## Step 10 — Test one fetch

In [34]:
if test_matches:
    m = test_matches[0]
    df_test = get_timeseries("225210", m["variable_code"])
    print(f"{m['parameter']}: {len(df_test)} rows")
    df_test.head()
else:
    print("No matches from Step 8 to test with.")

pH: 731 rows


## Step 11 — Checkpointing + logging helpers

In [35]:
def load_checkpoint() -> set:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return set(tuple(x) for x in json.load(f))
    return set()

def save_checkpoint(done: set):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(list(done), f)

def append_rows(df: pd.DataFrame):
    header = not os.path.exists(LONG_CSV)
    df.to_csv(LONG_CSV, mode="a", header=header, index=False)

def log_skip(station: str, reason: str):
    with open(SKIPPED_LOG, "a") as f:
        f.write(f"{datetime.now().isoformat()}  station={station}  reason={reason}\n")

def log_failure(station: str, parameter: str, error):
    with open(FAILURES_LOG, "a") as f:
        f.write(f"{datetime.now().isoformat()}  station={station}  parameter={parameter}  error={error}\n")

print("Helpers ready. Output CSV:", LONG_CSV)

Helpers ready. Output CSV: /Users/harsh/Desktop/wmis_export/wmis_all_stations_long.csv


## Step 12 — Pipeline function

Handles all skip cases explicitly and logs why, rather than silently dropping
stations. Uses the `candidate_stations_df` pre-filter from Step 6 by default,
which should give a much higher hit rate than scanning the full station list
blind.

In [36]:
def process_stations(station_ids, done):
    total_rows = 0
    skipped = 0
    for i, station in enumerate(station_ids, 1):
        print(f"[{i}/{len(station_ids)}] station {station}", end="  ")
        matches, skip_reason = match_parameters_for_station(station)

        meta = STATION_META.get(station, {})
        station_name = meta.get("station_name")
        latitude = meta.get("latitude")
        longitude = meta.get("longitude")

        if skip_reason:
            print(f"-> skipped ({skip_reason})")
            log_skip(station, skip_reason)
            skipped += 1
            time.sleep(REQUEST_DELAY)
            continue

        print(f"-> {len(matches)} parameter(s) matched")
        for m in matches:
            key = (station, m["parameter"])
            if key in done:
                continue
            try:
                df = get_timeseries(station, m["variable_code"], m["datasource"])
            except Exception as e:
                print(f"    {m['parameter']}: fetch failed ({e})")
                log_failure(station, m["parameter"], e)
                time.sleep(REQUEST_DELAY)
                continue

            if not df.empty:
                df["station"] = station
                df["station_name"] = station_name
                df["latitude"] = latitude
                df["longitude"] = longitude
                df["parameter"] = m["parameter"]
                df["variable_code"] = m["variable_code"]
                df["datasource"] = m["datasource"]
                df["matched_variable_name"] = m["matched_name"]
                df = df[["station", "station_name", "latitude", "longitude",
                          "parameter", "variable_code", "datasource",
                          "matched_variable_name", "datetime", "value", "quality_code"]]
                append_rows(df)
                total_rows += len(df)
                print(f"    {m['parameter']}: {len(df)} rows (running total {total_rows})")
            else:
                print(f"    {m['parameter']}: no data in date range")

            done.add(key)
            save_checkpoint(done)
            time.sleep(REQUEST_DELAY)

    return total_rows, skipped

print("process_stations ready")

process_stations ready


## Step 13 — Small test run (first 5 pre-filtered candidate stations)

In [37]:
test_stations = candidate_stations_df["station"].tolist()[:5]
print("Testing on:", test_stations)

done = load_checkpoint()
total_rows, skipped = process_stations(test_stations, done)

print(f"\nTest run done. {total_rows} rows written, {skipped} stations skipped.")
print("CSV exists:", os.path.exists(LONG_CSV))

Testing on: ['221001', '221201', '221202', '221203', '221204']
[1/5] station 221001  -> 6 parameter(s) matched
  retrying (1/3) after error: API error 126: No data within specified period
  retrying (2/3) after error: API error 126: No data within specified period
    pH: fetch failed (API error 126: No data within specified period)
  retrying (1/3) after error: API error 126: No data within specified period
  retrying (2/3) after error: API error 126: No data within specified period
    Streamflow: fetch failed (API error 126: No data within specified period)
  retrying (1/3) after error: API error 126: No data within specified period
  retrying (2/3) after error: API error 126: No data within specified period
    Stream Water Level: fetch failed (API error 126: No data within specified period)
  retrying (1/3) after error: API error 126: No data within specified period
  retrying (2/3) after error: API error 126: No data within specified period
    Salinity (EC): fetch failed (API er

## Step 15 — Batch run (10 stations at a time)

Re-run this cell to advance batch-by-batch through all pre-filtered candidate
stations. Safe to interrupt — completed (station, parameter) pairs are skipped
via the checkpoint file. Each run appends new rows to `wmis_batch_accumulated.csv`
(stations included even when only 1 of the 7 parameters has data).

In [38]:
def _periods_overlap(p_start, p_end, target_start, target_end):
    try:
        return not (p_end < target_start or p_start > target_end)
    except Exception:
        return True  # if unparseable, don't block the attempt - let the real fetch decide

def match_parameters_for_station(station: str):
    try:
        data = hydstra_request({
            "function": "get_variable_list",
            "version": 1,
            "params": {"site_list": station, "datasource": DATASOURCE},
        })
    except Exception as e:
        return [], f"get_variable_list error: {e}"

    sites = data.get("return", {}).get("sites", [])
    if not sites:
        return [], "no site entry returned"

    variables = sites[0].get("variables", [])
    if not variables:
        return [], "no WQ variables available at this station"

    available = {
        v["variable"]: v
        for v in variables
        if "variable" in v and v.get("subdesc", "").strip().lower() == "available for release"
    }

    if not available:
        return [], "WQ variables present but none marked 'Available for release'"

    matches = []
    for param, code_ in CONFIRMED_PARAM_CODES.items():
        if code_ not in available:
            continue
        v = available[code_]
        p_start, p_end = v.get("period_start", ""), v.get("period_end", "")
        if p_start and p_end and not _periods_overlap(p_start, p_end, START_TIME, END_TIME):
            continue  # variable exists but has no data in our date range - skip, don't fetch
        matches.append({
            "parameter": param,
            "variable_code": code_,
            "datasource": DATASOURCE,
            "matched_name": v.get("name", ""),
        })

    if not matches:
        return [], "WQ data present but nothing overlaps target date range / none of the 7 params found"

    return matches, None

print("match_parameters_for_station updated with date-range filtering")

match_parameters_for_station updated with date-range filtering


In [39]:
def _load_long_csv_safe():
    expected_cols = ["station", "parameter", "variable_code", "datasource",
                      "matched_variable_name", "datetime", "value", "quality_code"]
    if os.path.exists(LONG_CSV) and os.path.getsize(LONG_CSV) > 0:
        try:
            df = pd.read_csv(LONG_CSV)
            if "station" in df.columns:
                return df
            print(f"WARNING: {LONG_CSV} exists but doesn't have a 'station' column "
                  f"(columns found: {df.columns.tolist()}) - likely a stale file from "
                  f"an earlier run. Treating as empty.")
        except Exception as e:
            print(f"WARNING: couldn't read {LONG_CSV} ({e}) - treating as empty.")
    return pd.DataFrame(columns=expected_cols)

In [41]:
BATCH_SIZE = 20
BATCH_STATE_FILE = os.path.join(OUTPUT_DIR, "current_batch.json")

def _load_long_csv_safe():
    expected_cols = ["station", "station_name", "latitude", "longitude",
                      "parameter", "variable_code", "datasource",
                      "matched_variable_name", "datetime", "value", "quality_code"]
    if os.path.exists(LONG_CSV) and os.path.getsize(LONG_CSV) > 0:
        try:
            df = pd.read_csv(LONG_CSV)
            if "station" in df.columns:
                return df
            print(f"WARNING: {LONG_CSV} exists but doesn't have a 'station' column "
                  f"(columns found: {df.columns.tolist()}) - likely a stale file from "
                  f"an earlier run. Treating as empty.")
        except Exception as e:
            print(f"WARNING: couldn't read {LONG_CSV} ({e}) - treating as empty.")
    return pd.DataFrame(columns=expected_cols)

# --- everything below this line in your existing Step 15 cell is unchanged ---

all_candidate_stations = candidate_stations_df["station"].tolist()
n_batches = (len(all_candidate_stations) - 1) // BATCH_SIZE + 1
print(f"Full run: {len(all_candidate_stations)} candidate stations, {n_batches} batches of {BATCH_SIZE}")

# Track which batch to run next so re-running this cell advances automatically
# instead of restarting from batch 1.
if os.path.exists(BATCH_STATE_FILE):
    with open(BATCH_STATE_FILE) as f:
        next_batch_num = json.load(f)["next_batch"]
else:
    next_batch_num = 1

if next_batch_num > n_batches:
    print(f"\nAll {n_batches} batches already completed. Nothing left to run.")
else:
    batch_start = (next_batch_num - 1) * BATCH_SIZE
    batch_stations = all_candidate_stations[batch_start:batch_start + BATCH_SIZE]
    batch_end = batch_start + len(batch_stations)

    print(f"\n=== Running batch {next_batch_num}/{n_batches}: "
          f"stations {batch_start + 1}-{batch_end} ===")

    done = load_checkpoint()
    total_rows, skipped = process_stations(batch_stations, done)
    print(f"\nBatch {next_batch_num} done. {total_rows} rows written this batch, {skipped} stations skipped.")

    # Per-station check: did we actually get data for each station in this batch?
    combined_so_far = _load_long_csv_safe()
    combined_so_far["station"] = combined_so_far["station"].astype(str)

    print(f"\n--- Per-station data check (batch {next_batch_num}) ---")
    stations_with_data = []
    stations_without_data = []
    for station in batch_stations:
        n_rows = len(combined_so_far[combined_so_far["station"] == station])
        if n_rows > 0:
            print(f"  {station}: {n_rows} rows extracted")
            stations_with_data.append(station)
        else:
            print(f"  {station}: NO DATA extracted")
            stations_without_data.append(station)

    print(f"\n{len(stations_with_data)}/{len(batch_stations)} stations in this batch have data, "
          f"{len(stations_without_data)} have none (see {SKIPPED_LOG} / {FAILURES_LOG} for why).")

    # Save this batch's data to its own named CSV
    batch_rows = combined_so_far[combined_so_far["station"].isin(batch_stations)]
    batch_csv_path = os.path.join(OUTPUT_DIR, f"batch {next_batch_num} data.csv")
    batch_rows.to_csv(batch_csv_path, index=False)
    print(f"\nSaved: {batch_csv_path} ({len(batch_rows)} rows)")

    # Advance the counter for next time this cell is run
    with open(BATCH_STATE_FILE, "w") as f:
        json.dump({"next_batch": next_batch_num + 1}, f)

    print(f"\nBatch {next_batch_num} complete. Re-run this cell to process batch {next_batch_num + 1} "
          f"({'none remaining' if next_batch_num >= n_batches else f'{n_batches - next_batch_num} batches left'}).")

Full run: 1460 candidate stations, 73 batches of 20

=== Running batch 11/73: stations 201-220 ===
[1/20] station 226606  -> skipped (WQ data present but nothing overlaps target date range / none of the 7 params found)
[2/20] station 226607  -> skipped (no WQ variables available at this station)
[3/20] station 226608  -> skipped (no WQ variables available at this station)
[4/20] station 226609  -> skipped (no WQ variables available at this station)
[5/20] station 226610  -> skipped (no WQ variables available at this station)
[6/20] station 226611  -> skipped (no WQ variables available at this station)
[7/20] station 226612  -> skipped (WQ data present but nothing overlaps target date range / none of the 7 params found)
[8/20] station 226700  -> skipped (no WQ variables available at this station)
[9/20] station 226702  -> skipped (no WQ variables available at this station)
[10/20] station 227001  -> skipped (no WQ variables available at this station)
[11/20] station 227200  -> 6 paramet

/var/folders/nn/7hzwkz613z3d2qzr9qzs3gs80000gn/T/ipykernel_81900/1328977790.py:10: DtypeWarning: Columns (0: parameter, 1: datetime, 2: value, 3: quality_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(LONG_CSV)
